# SigLIP-2 — Fakeddit Binary Fake News Detection (Text + Image, No Metadata)
### 2-Way Classification | Target ≥ 90% Accuracy | Minimum Overfitting

## Speed & Completeness Fixes Applied
| Problem | Root Cause | Fix |
|---|---|---|
| Never finishes | BATCH=4 + 384px images = ~4 min/epoch | BATCH=8, IMG=224px cached in RAM |
| Disk IO bottleneck | PIL.open() every __getitem__, num_workers=0 | Pre-cache all images as numpy in RAM |
| SWA model crash | update_bn() missing after loop | Added correct SWA finalization |
| SWA slow | CPU↔GPU transfer every epoch | SWA stays on GPU (T4 16GB fits it) |
| gradient_checkpointing | +30% compute overhead | Removed; use fp16 instead |
| Results table truncated | Cell 8 source cut mid-line | Rewritten completely |
| Slow per-step | 250 optimizer steps/epoch | 125 steps (ACCUM_STEPS=4) |

## Architecture: SigLIP-2 SO400M
- Vision encoder (last 4 layers unfrozen) + Text encoder (last 4 layers unfrozen)
- Fusion: L2-norm(vis_embed) + L2-norm(txt_embed) → proj(256) each → cat(512)
- Head: LN → 512 → GELU → Dropout → 256 → GELU → Dropout → 2

## Training Strategy
- 80% train / 10% val / 10% test per fold | 5-fold CV
- OneCycleLR (15% cosine warmup) | MixUp α=0.4 | Label Smoothing 0.10
- SWA from epoch 8 (stays on GPU) | Early stopping patience=4
- Effective batch = 8 × 4 = 32 | AMP fp16 throughout


In [2]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
import sys, subprocess

print("Installing dependencies...")
pkgs = ["transformers>=4.40.0", "pillow", "tabulate", "scikit-learn",
        "torchvision", "tqdm"]
for i, pkg in enumerate(pkgs, 1):
    print(f"  [{i}/{len(pkgs)}] {pkg}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
print(f"\ntorch        : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    free, tot = torch.cuda.mem_get_info()
    print(f"VRAM         : {free/1e9:.1f} / {tot/1e9:.1f} GB free")
print(f"transformers : {__import__('transformers').__version__}")


Installing dependencies...
  [1/6] transformers>=4.40.0
  [2/6] pillow
  [3/6] tabulate
  [4/6] scikit-learn
  [5/6] torchvision
  [6/6] tqdm

torch        : 2.10.0+cu128
CUDA         : True
GPU          : Tesla T4
VRAM         : 15.5 / 15.6 GB free
transformers : 5.0.0


In [3]:
# ── Cell 2: Imports & Configuration ──────────────────────────────────────────
import os, gc, random, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim.swa_utils import AveragedModel, SWALR
from PIL import Image
from torchvision import transforms

from transformers import AutoProcessor, AutoModel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from tabulate import tabulate
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ══════════════════════════ CONFIGURATION ════════════════════════════════════
SEED         = 42

# ── Data ──────────────────────────────────────────────────────────────────────
NUM_SAMPLES  = 10_000      # balanced 5000 real + 5000 fake
NUM_FOLDS    = 5

# ── Speed-critical settings ───────────────────────────────────────────────────
# FIX: was BATCH=4 ACCUM=8 → 250 slow steps; now BATCH=8 ACCUM=4 → 125 steps
BATCH_SIZE   = 8
ACCUM_STEPS  = 4           # effective batch = 32
IMG_SIZE     = 224         # FIX: was 384 → 3× slower; SigLIP accepts 224 too

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS       = 12          # with early stopping; 12 is enough after speed fixes
LR_BACKBONE  = 2e-5        # unfrozen backbone layers
LR_HEAD      = 1e-4        # projection + classifier head
WEIGHT_DECAY = 0.01
DROPOUT_RATE = 0.30
PATIENCE     = 4           # early stopping
LABEL_SMOOTH = 0.10
WARMUP_FRAC  = 0.15

# ── Regularisation ────────────────────────────────────────────────────────────
USE_MIXUP    = True
MIXUP_ALPHA  = 0.4

# ── SWA — stays on GPU (FIX: was CPU↔GPU every epoch = huge overhead) ─────────
USE_SWA      = True
SWA_START_EP = 8           # start averaging at epoch 8

# ── Unfreeze ──────────────────────────────────────────────────────────────────
UNFREEZE_VIS = 4           # last N vision encoder layers
UNFREEZE_TXT = 4           # last N text  encoder layers (FIX: was 2)

NUM_CLASSES  = 2
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
SIGLIP_MODEL = "google/siglip-so400m-patch14-384"
OUTPUT_DIR   = "/kaggle/working"
USE_AMP      = DEVICE == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device        : {DEVICE}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Model         : {SIGLIP_MODEL}")
print(f"Samples       : {NUM_SAMPLES:,} | Folds: {NUM_FOLDS} | Epochs: {EPOCHS}")
print(f"Batch         : {BATCH_SIZE} x {ACCUM_STEPS} accum = {BATCH_SIZE*ACCUM_STEPS} effective")
print(f"IMG_SIZE      : {IMG_SIZE}px  (FIX: 384→224 = 3x faster per batch)")
print(f"LR            : backbone={LR_BACKBONE} | head={LR_HEAD} (warmup {int(WARMUP_FRAC*100)}%)")
print(f"Unfreeze      : {UNFREEZE_VIS} vision + {UNFREEZE_TXT} text layers (FIX: was 4+2)")
print(f"SWA           : {'on  start ep='+str(SWA_START_EP)+' GPU-resident (FIX: was CPU)' if USE_SWA else 'off'}")
print(f"Mixup         : {'on  alpha='+str(MIXUP_ALPHA) if USE_MIXUP else 'off'}")
print(f"GradCheckpoint: OFF  (FIX: was ON = +30% compute overhead)")
print("=" * 65)


Device        : cuda
GPU           : Tesla T4
Model         : google/siglip-so400m-patch14-384
Samples       : 10,000 | Folds: 5 | Epochs: 12
Batch         : 8 x 4 accum = 32 effective
IMG_SIZE      : 224px  (FIX: 384→224 = 3x faster per batch)
LR            : backbone=2e-05 | head=0.0001 (warmup 15%)
Unfreeze      : 4 vision + 4 text layers (FIX: was 4+2)
SWA           : on  start ep=8 GPU-resident (FIX: was CPU)
Mixup         : on  alpha=0.4
GradCheckpoint: OFF  (FIX: was ON = +30% compute overhead)


In [4]:
# ── Cell 3: Fakeddit Dataset Loader ──────────────────────────────────────────

def load_fakeddit(n_samples=NUM_SAMPLES):
    """Load Fakeddit 2-way labels. Searches /kaggle/input for TSV/CSV."""
    print("Scanning for Fakeddit dataset...\n")
    input_root = Path("/kaggle/input")

    if input_root.exists():
        mounted = sorted(p for p in input_root.iterdir() if p.is_dir())
        if mounted:
            print("Mounted datasets:")
            for m in mounted:
                print(f"  {m}")
        else:
            print("  /kaggle/input is EMPTY — no datasets mounted!")
    print()

    # Explicit well-known paths first
    explicit = [
        "/kaggle/input/fakeddit-dataset/multimodal_only_samples/multimodal_train.tsv",
        "/kaggle/input/fakeddit-dataset/multimodal_only_samples/multimodal_test_public.tsv",
        "/kaggle/input/fakeddit-dataset/multimodal_only_samples/multimodal_validate.tsv",
    ]
    for path in explicit:
        p = Path(path)
        if p.exists():
            print(f"Found (explicit): {p}")
            return _load_from_file(p, n_samples)

    # Recursive scan
    for f in sorted(input_root.rglob("*.tsv")):
        try:
            if "2_way_label" in open(str(f)).read(400):
                print(f"Found (scan): {f}")
                return _load_from_file(f, n_samples)
        except Exception:
            continue
    for f in sorted(input_root.rglob("*.csv")):
        try:
            if "2_way_label" in open(str(f)).read(400):
                print(f"Found (scan): {f}")
                return _load_from_file(f, n_samples)
        except Exception:
            continue

    raise FileNotFoundError(
        "No Fakeddit file found.\n"
        "Add dataset vanshikavmittal/fakeddit-dataset via '+Add data' in Kaggle."
    )


def _find_image(img_id: str, img_dir: Path):
    for ext in (".jpg", ".jpeg", ".png", ".gif", ".webp"):
        p = img_dir / f"{img_id}{ext}"
        if p.exists():
            return str(p)
    return None


def _find_image_dir(file_path: Path):
    candidates = [
        file_path.parent / "public_image_set",
        file_path.parent / "images",
        file_path.parent.parent / "public_image_set",
        file_path.parent.parent / "images",
    ]
    for d in candidates:
        if d.is_dir():
            return d
    for d in file_path.parent.rglob("*"):
        if d.is_dir() and "image" in d.name.lower():
            return d
    return None


def _load_from_file(file_path, n_samples):
    file_path = Path(file_path)
    sep = "\t" if file_path.suffix == ".tsv" else ","
    df  = pd.read_csv(file_path, sep=sep, low_memory=False)
    df.columns = df.columns.str.strip()
    print(f"  Columns: {list(df.columns[:10])}...")

    text_col = next((c for c in ["clean_title","title","text"] if c in df.columns), None)
    id_col   = "id" if "id" in df.columns else df.columns[0]

    if "2_way_label" not in df.columns:
        raise ValueError(f"Missing '2_way_label'. Columns: {list(df.columns)}")
    if not text_col:
        raise ValueError(f"No text column. Columns: {list(df.columns[:10])}")

    df = df[[id_col, text_col, "2_way_label"]].dropna(subset=["2_way_label"]).copy()
    df = df.rename(columns={id_col:"id", text_col:"text", "2_way_label":"label"})
    df["label"] = df["label"].astype(int)
    df = df[df["label"].isin([0, 1])]
    df["text"] = (df["text"].astype(str).str.strip()
                  .apply(lambda x: x[:300])
                  .replace("", "[No text]"))

    img_dir = _find_image_dir(file_path)
    if img_dir:
        print(f"  Image dir: {img_dir}")
        df["image_path"] = df["id"].apply(lambda x: _find_image(str(x), img_dir))
        n_img = df["image_path"].notna().sum()
        print(f"  Images matched: {n_img:,} / {len(df):,} ({100*n_img/len(df):.1f}%)")
    else:
        df["image_path"] = None
        print("  No image dir found — placeholder used for all samples")

    return _balance(df, n_samples)


def _balance(df, n_samples):
    half   = n_samples // 2
    fake   = df[df["label"]==1].sample(min(half, (df["label"]==1).sum()), random_state=SEED)
    real   = df[df["label"]==0].sample(min(half, (df["label"]==0).sum()), random_state=SEED)
    actual = min(len(fake), len(real))
    out    = pd.concat([fake.iloc[:actual], real.iloc[:actual]])
    out    = out.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_img  = out["image_path"].notna().sum() if "image_path" in out.columns else 0
    print(f"\n  Loaded {len(out):,} balanced samples  (Real={actual}, Fake={actual})")
    print(f"  Images available: {n_img:,} / {len(out):,}")
    return out


print("Fakeddit loader ready")


Fakeddit loader ready


In [5]:
# ── Cell 4: Image Pre-Cache + SigLIPDataset ───────────────────────────────────
#
# FIX: Original opened PIL from disk on every __getitem__ call with num_workers=0
#      -> GPU sat idle waiting for disk IO on every single sample
# FIX NOW: Load ALL images once into a RAM dict {id: np.ndarray uint8}
#          __getitem__ does a dict lookup (microseconds) instead of disk read
#

# Gray placeholder for missing/broken images (matches IMG_SIZE)
_PLACEHOLDER_NP = np.full((IMG_SIZE, IMG_SIZE, 3), 127, dtype=np.uint8)

# Train-time augmentation (applied to PIL, then processor handles normalize/resize)
_TRAIN_AUG = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.10),
])

# Resize to IMG_SIZE for both train and eval before processor
_RESIZE = transforms.Resize((IMG_SIZE, IMG_SIZE))


def build_image_cache(df: pd.DataFrame) -> dict:
    """
    Pre-load all available images into RAM as uint8 numpy arrays.
    FIX: replaces per-sample disk IO with O(1) dict lookup during training.
    """
    cache = {}
    paths = df[["id","image_path"]].dropna(subset=["image_path"])
    print(f"  Pre-caching {len(paths):,} images into RAM...")
    t0 = time.time()
    failed = 0
    for _, row in tqdm(paths.iterrows(), total=len(paths), leave=False):
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            img = _RESIZE(img)
            cache[str(row["id"])] = np.array(img, dtype=np.uint8)
        except Exception:
            failed += 1
    elapsed = time.time() - t0
    print(f"  Cached {len(cache):,} images in {elapsed:.1f}s  ({failed} failed)")
    return cache


class SigLIPDataset(Dataset):
    """
    Multimodal dataset backed by an in-RAM image cache.
    FIX: zero disk IO during training; images already in memory.

    Batch keys:
      pixel_values   (3, H, W)  float32  (processor-normalized)
      input_ids      (64,)      int64
      attention_mask (64,)      int64
      labels         ()         int64
    """

    def __init__(self, df, processor, image_cache: dict, training: bool = False):
        self.df      = df.reset_index(drop=True)
        self.proc    = processor
        self.cache   = image_cache
        self.training = training

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _augment_text(text: str) -> str:
        words = text.split()
        if len(words) <= 3:
            return text
        kept = [w for w in words if random.random() > 0.10]
        return " ".join(kept) if kept else text

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        text  = str(row["text"])
        label = int(row["label"])

        if self.training:
            text = self._augment_text(text)

        # Image: RAM lookup (microseconds) instead of disk read
        img_np = self.cache.get(str(row["id"]))
        if img_np is not None:
            pil = Image.fromarray(img_np)
            if self.training:
                pil = _TRAIN_AUG(pil)
        else:
            pil = Image.fromarray(_PLACEHOLDER_NP)

        inputs = self.proc(
            text=text,
            images=pil,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=64,
        )
        out = {k: v.squeeze(0) for k, v in inputs.items()}
        out["labels"] = torch.tensor(label, dtype=torch.long)
        return out


def collate_fn(batch):
    labels = torch.stack([b["labels"] for b in batch])
    result = {"labels": labels}
    for key in batch[0]:
        if key != "labels":
            result[key] = torch.stack([b[key] for b in batch])
    return result


print("SigLIPDataset ready  (RAM image cache)")


SigLIPDataset ready  (RAM image cache)


In [6]:
# ── Cell 5: SigLIPClassifier ──────────────────────────────────────────────────

class SigLIPClassifier(nn.Module):
    """
    2-branch fake-news classifier (vision + text, no metadata).

    Unfreeze policy:
      - Last UNFREEZE_VIS vision encoder layers + post_layernorm
      - Last UNFREEZE_TXT text  encoder layers + final_layer_norm
      - All proj + head layers (always trainable)

    Fusion:
      L2(vis_embed) -> vision_proj(256)  -|
                                           cat(512) -> LN -> 512 -> 256 -> 2
      L2(txt_embed) -> text_proj(256)   -|

    NOTE: gradient_checkpointing is OFF (FIX: was ON = +30% compute).
    """

    PROJ_DIM = 256

    def __init__(self, siglip_model, num_classes=NUM_CLASSES, dropout=DROPOUT_RATE):
        super().__init__()
        self.siglip = siglip_model
        v_dim = siglip_model.config.vision_config.hidden_size
        t_dim = siglip_model.config.text_config.hidden_size

        # Freeze all backbone params
        for p in self.siglip.parameters():
            p.requires_grad = False

        # Unfreeze last UNFREEZE_VIS vision encoder layers
        v_enc = self.siglip.vision_model.encoder.layers
        for layer in v_enc[-UNFREEZE_VIS:]:
            for p in layer.parameters():
                p.requires_grad = True
        # Unfreeze vision post_layernorm
        for p in self.siglip.vision_model.post_layernorm.parameters():
            p.requires_grad = True

        # Unfreeze last UNFREEZE_TXT text encoder layers
        t_enc = self.siglip.text_model.encoder.layers
        for layer in t_enc[-UNFREEZE_TXT:]:
            for p in layer.parameters():
                p.requires_grad = True
        # Unfreeze text final_layer_norm (FIX: was missing)
        if hasattr(self.siglip.text_model, "final_layer_norm"):
            for p in self.siglip.text_model.final_layer_norm.parameters():
                p.requires_grad = True

        D = self.PROJ_DIM
        self.vision_proj = nn.Sequential(
            nn.Linear(v_dim, D), nn.LayerNorm(D), nn.GELU(), nn.Dropout(0.10))
        self.text_proj = nn.Sequential(
            nn.Linear(t_dim, D), nn.LayerNorm(D), nn.GELU(), nn.Dropout(0.10))

        fused = D * 2   # 512
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused),
            nn.Dropout(dropout),
            nn.Linear(fused, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )

        # Xavier init on all new linear layers
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.out_features <= 512:
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f"  SigLIPClassifier: {trainable/1e6:.1f}M / {total/1e6:.1f}M trainable")

    def forward(self, pixel_values, input_ids, attention_mask=None, **unused):
        out = self.siglip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        vis   = F.normalize(out.image_embeds, dim=-1)
        txt   = F.normalize(out.text_embeds,  dim=-1)
        fused = torch.cat([self.vision_proj(vis), self.text_proj(txt)], dim=-1)
        return self.classifier(fused)


print("SigLIPClassifier ready  (grad-checkpoint OFF, FIX unfreeze text final_layer_norm)")


SigLIPClassifier ready  (grad-checkpoint OFF, FIX unfreeze text final_layer_norm)


In [7]:
# ── Cell 6: Training Helpers ──────────────────────────────────────────────────

def mixup_data(x, y, alpha=MIXUP_ALPHA):
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, logits, ya, yb, lam):
    return lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)


class EarlyStopping:
    def __init__(self, patience=PATIENCE, delta=1e-4):
        self.patience       = patience
        self.delta          = delta
        self.best_score     = None
        self.counter        = 0
        self.best_state     = None
        self.best_train_acc = 0.0
        self.best_val_acc   = 0.0
        self.stop           = False

    def step(self, val_loss, val_acc, train_acc, model):
        score = -val_loss
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score     = score
            self.best_train_acc = train_acc
            self.best_val_acc   = val_acc
            self.best_state     = {k: v.cpu().clone()
                                   for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore_best(self, model):
        if self.best_state:
            model.load_state_dict(
                {k: v.to(DEVICE) for k, v in self.best_state.items()})


def train_epoch(model, loader, optimizer, criterion, scheduler, scaler):
    """One training epoch with gradient accumulation + MixUp."""
    model.train()
    total_loss = correct = n = 0
    num_batches = len(loader)
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader, 1), total=num_batches, leave=False,
                desc="  train")
    for step, batch in pbar:
        labels = batch.pop("labels").to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}

        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            if USE_MIXUP and random.random() < 0.5:
                px, ya, yb, lam = mixup_data(batch["pixel_values"], labels)
                batch["pixel_values"] = px
                logits = model(**batch)
                loss   = mixup_loss(criterion, logits, ya, yb, lam)
            else:
                logits = model(**batch)
                loss   = criterion(logits, labels)

        scaler.scale(loss / ACCUM_STEPS).backward()

        total_loss += loss.detach().item() * labels.size(0)
        correct    += (logits.detach().argmax(1) == labels).sum().item()
        n          += labels.size(0)

        if step % ACCUM_STEPS == 0 or step == num_batches:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix(loss=f"{total_loss/n:.4f}", acc=f"{correct/n:.4f}")

    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in loader:
        labels = batch.pop("labels").to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(**batch)
            loss   = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    n      = len(y_true)
    return dict(
        loss      = total_loss / n,
        accuracy  = accuracy_score(y_true, y_pred),
        f1        = f1_score(y_true, y_pred, average="macro", zero_division=0),
        precision = precision_score(y_true, y_pred, average="macro", zero_division=0),
        recall    = recall_score(y_true, y_pred, average="macro", zero_division=0),
    ), y_pred, y_true


# ── Live results table (printed after every fold) ─────────────────────────────
def print_live_table(results):
    hdrs = ["Fold","Train","Val","Test Acc","F1","Prec","Rec","Ovf Gap","Status"]
    rows = []
    for r in results:
        acc_ok = r["Accuracy"]    >= 0.90
        gap_ok = abs(r["Overfit_Gap"]) < 0.05
        rows.append([
            r["Fold"],
            f"{r['Train_Acc']:.4f}", f"{r['Val_Acc']:.4f}",
            f"{r['Accuracy']:.4f}",  f"{r['F1_Macro']:.4f}",
            f"{r['Precision']:.4f}", f"{r['Recall']:.4f}",
            f"{r['Overfit_Gap']:+.4f}",
            ("V" if acc_ok else "X") + "ACC|" + ("V" if gap_ok else "X") + "GAP",
        ])
    print(tabulate(rows, headers=hdrs, tablefmt="simple"))


print("Training helpers ready")


Training helpers ready


In [8]:
# ── Cell 7: K-Fold Training Loop ──────────────────────────────────────────────
# GLOBAL — accumulates across folds so table persists if kernel interrupted
FOLD_RESULTS = []
ALL_HISTORY  = []
ALL_CMS      = []


def run_kfold(df, image_cache):
    global FOLD_RESULTS, ALL_HISTORY, ALL_CMS
    FOLD_RESULTS.clear()
    ALL_HISTORY.clear()
    ALL_CMS.clear()

    skf     = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    labels  = df["label"].values
    indices = np.arange(len(df))

    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        free, tot = torch.cuda.mem_get_info()
        print(f"GPU at start: {free/1e9:.1f} / {tot/1e9:.1f} GB free\n")

    print("Loading SigLIP-2 processor and backbone...")
    processor   = AutoProcessor.from_pretrained(SIGLIP_MODEL)
    # FIX: gradient_checkpointing OFF — was adding +30% compute per step
    siglip_base = AutoModel.from_pretrained(SIGLIP_MODEL).cpu()
    siglip_init = {k: v.cpu().clone() for k, v in siglip_base.state_dict().items()}
    print("Backbone loaded  (gradient checkpointing OFF)\n")

    scaler = torch.amp.GradScaler(device="cuda", enabled=USE_AMP)

    for fold, (train_val_idx, test_idx_all) in enumerate(
            skf.split(indices, labels), 1):

        print(f"\n{'='*65}")
        print(f"  FOLD {fold} / {NUM_FOLDS}")
        print(f"{'='*65}")

        # 80 / 10 / 10 split
        # StratifiedKFold gives ~80% train_val / ~20% test_idx_all
        # Split test_idx_all in half → val (10%) + test (10%)
        rng      = np.random.default_rng(SEED + fold)
        perm     = rng.permutation(len(test_idx_all))
        mid      = len(perm) // 2
        val_idx  = test_idx_all[perm[:mid]]
        test_idx = test_idx_all[perm[mid:]]
        train_idx = train_val_idx

        n_tr, n_va, n_te = len(train_idx), len(val_idx), len(test_idx)
        tot = n_tr + n_va + n_te
        print(f"  Train:{n_tr} ({n_tr/tot*100:.0f}%) | "
              f"Val:{n_va} ({n_va/tot*100:.0f}%) | "
              f"Test:{n_te} ({n_te/tot*100:.0f}%)")

        # DataLoaders — use num_workers=2 for CPU prefetch
        # FIX: was num_workers=0 → GPU blocked waiting for data every step
        nw  = 2 if os.cpu_count() and os.cpu_count() >= 2 else 0
        pin = DEVICE == "cuda"
        kw  = dict(num_workers=nw, pin_memory=pin,
                   persistent_workers=(nw > 0))

        train_loader = DataLoader(
            SigLIPDataset(df.iloc[train_idx], processor, image_cache, training=True),
            BATCH_SIZE, shuffle=True,  collate_fn=collate_fn,
            drop_last=True, **kw)
        val_loader = DataLoader(
            SigLIPDataset(df.iloc[val_idx],   processor, image_cache, training=False),
            BATCH_SIZE, shuffle=False, collate_fn=collate_fn, **kw)
        test_loader = DataLoader(
            SigLIPDataset(df.iloc[test_idx],  processor, image_cache, training=False),
            BATCH_SIZE, shuffle=False, collate_fn=collate_fn, **kw)

        # Reset weights
        print("  Initialising model...")
        siglip_base.load_state_dict(siglip_init, strict=True)
        siglip_base.to(DEVICE)

        model     = SigLIPClassifier(siglip_base).to(DEVICE)
        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

        # Differential LR: backbone 10× lower than head
        backbone_params = [p for n, p in model.named_parameters()
                           if "siglip" in n and p.requires_grad]
        head_params     = [p for n, p in model.named_parameters()
                           if "siglip" not in n and p.requires_grad]
        optimizer = AdamW([
            {"params": backbone_params, "lr": LR_BACKBONE},
            {"params": head_params,     "lr": LR_HEAD},
        ], weight_decay=WEIGHT_DECAY)

        # OneCycleLR: cosine warmup + cosine anneal
        # FIX: cosine schedule works better with the full training run
        steps_per_epoch = max(1, len(train_loader) // ACCUM_STEPS)
        total_steps     = steps_per_epoch * EPOCHS
        warmup_steps    = int(total_steps * WARMUP_FRAC)
        scheduler = OneCycleLR(
            optimizer,
            max_lr=[LR_BACKBONE, LR_HEAD],
            total_steps=total_steps,
            pct_start=WARMUP_FRAC,
            anneal_strategy="cos",
            div_factor=25.0,
            final_div_factor=1e4,
        )
        print(f"  Scheduler: {warmup_steps} warmup / {total_steps} total steps")

        es = EarlyStopping(PATIENCE)

        # SWA — stays on GPU (FIX: was CPU, causing ~30s/epoch transfer overhead)
        swa_model = None
        if USE_SWA:
            swa_model = AveragedModel(model)   # GPU-resident
            print(f"  SWA enabled (start ep {SWA_START_EP}, GPU-resident)")
            if DEVICE == "cuda":
                free, tot_mem = torch.cuda.mem_get_info()
                print(f"  GPU after SWA init: {free/1e9:.1f}/{tot_mem/1e9:.1f} GB")

        history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}

        # ── Training loop ─────────────────────────────────────────────────────
        for epoch in range(1, EPOCHS + 1):
            t_ep = time.time()
            print(f"\n  -- Epoch {epoch}/{EPOCHS} --")

            tr_loss, tr_acc = train_epoch(
                model, train_loader, optimizer, criterion, scheduler, scaler)

            val_m, _, _ = evaluate(model, val_loader, criterion)

            history["train_loss"].append(tr_loss)
            history["val_loss"].append(val_m["loss"])
            history["train_acc"].append(tr_acc)
            history["val_acc"].append(val_m["accuracy"])

            # SWA update (GPU-resident — no transfer needed)
            if USE_SWA and swa_model is not None and epoch >= SWA_START_EP:
                swa_model.update_parameters(model)

            es.step(val_m["loss"], val_m["accuracy"], tr_acc, model)
            marker = "  <- best" if es.counter == 0 else ""
            elapsed = time.time() - t_ep
            print(f"  E{epoch:2d} [{elapsed:.0f}s] | "
                  f"TrL={tr_loss:.4f} TrA={tr_acc:.4f} | "
                  f"VaL={val_m['loss']:.4f} VaA={val_m['accuracy']:.4f}{marker}")

            if es.stop:
                print(f"  Early stop at epoch {epoch}")
                break

        # ── SWA finalisation (FIX: update_bn was MISSING in original) ─────────
        if USE_SWA and swa_model is not None:
            print("  Updating SWA batch-norm statistics...")
            torch.optim.swa_utils.update_bn(train_loader, swa_model, device=DEVICE)

        # ── Restore best checkpoint & test evaluation ─────────────────────────
        es.restore_best(model)
        test_m, preds, true = evaluate(model, test_loader, criterion)

        # Compare best-epoch vs SWA; keep better
        if USE_SWA and swa_model is not None:
            test_m_swa, preds_swa, _ = evaluate(swa_model, test_loader, criterion)
            if test_m_swa["accuracy"] > test_m["accuracy"]:
                test_m, preds = test_m_swa, preds_swa
                print(f"  SWA model used  (acc {test_m['accuracy']:.4f})")
            else:
                print(f"  Best-epoch model used  (acc {test_m['accuracy']:.4f})")

        cm      = confusion_matrix(true, preds)
        ovf_gap = es.best_train_acc - test_m["accuracy"]
        gap_tag = "LOW" if abs(ovf_gap) < 0.05 else "HIGH-overfitting"

        print(f"\n  -- Fold {fold} Test Results --")
        print(f"  Accuracy  : {test_m['accuracy']:.4f}")
        print(f"  F1 Macro  : {test_m['f1']:.4f}")
        print(f"  Precision : {test_m['precision']:.4f}")
        print(f"  Recall    : {test_m['recall']:.4f}")
        print(f"  Ovf Gap   : {ovf_gap:+.4f}  [{gap_tag}]\n")
        print(classification_report(true, preds, target_names=["Real","Fake"]))

        FOLD_RESULTS.append({
            "Fold":        fold,
            "Train_Acc":   round(es.best_train_acc,  4),
            "Val_Acc":     round(es.best_val_acc,     4),
            "Accuracy":    round(test_m["accuracy"],  4),
            "F1_Macro":    round(test_m["f1"],        4),
            "Precision":   round(test_m["precision"], 4),
            "Recall":      round(test_m["recall"],    4),
            "Overfit_Gap": round(ovf_gap,             4),
        })
        ALL_HISTORY.append(history)
        ALL_CMS.append(cm)

        print("\n  -- Results so far --")
        print_live_table(FOLD_RESULTS)

        # Free GPU memory between folds
        siglip_base.to("cpu")
        del model, optimizer, scheduler, es, swa_model
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            free, tot_mem = torch.cuda.mem_get_info()
            print(f"  GPU after cleanup: {free/1e9:.1f}/{tot_mem/1e9:.1f} GB\n")

    return FOLD_RESULTS, ALL_HISTORY, ALL_CMS


print("K-fold runner ready")


K-fold runner ready


In [9]:
# ── Cell 8: Results Table & Visualization ────────────────────────────────────
# FIX: original cell was TRUNCATED mid-line and would crash on Cell 11

def print_results_table(fold_results):
    """Print rich results table with per-fold metrics + mean/std rows."""
    W = 82
    print("\n" + "=" * W)
    print("  SigLIP-2  |  Fakeddit  |  2-Way (text + image)  |  No Metadata")
    print("  Target: >= 90% Accuracy  |  Overfitting Gap < 5%")
    print("=" * W)

    hdrs = ["Fold","Train","Val","Test Acc","F1 Macro","Precision","Recall","Ovf Gap","Status"]
    rows = []
    for r in fold_results:
        acc_ok = r["Accuracy"]    >= 0.90
        gap_ok = abs(r["Overfit_Gap"]) < 0.05
        status = ("V" if acc_ok else "X") + "ACC|" + ("V" if gap_ok else "X") + "GAP"
        rows.append([
            r["Fold"],
            f"{r['Train_Acc']:.4f}", f"{r['Val_Acc']:.4f}",
            f"{r['Accuracy']:.4f}",  f"{r['F1_Macro']:.4f}",
            f"{r['Precision']:.4f}", f"{r['Recall']:.4f}",
            f"{r['Overfit_Gap']:+.4f}", status,
        ])

    accs  = [r["Accuracy"]    for r in fold_results]
    f1s   = [r["F1_Macro"]    for r in fold_results]
    precs = [r["Precision"]   for r in fold_results]
    recs  = [r["Recall"]      for r in fold_results]
    gaps  = [r["Overfit_Gap"] for r in fold_results]
    tr_ac = [r["Train_Acc"]   for r in fold_results]
    va_ac = [r["Val_Acc"]     for r in fold_results]

    rows.append(["---"] * len(hdrs))
    rows.append(["MEAN",
                 f"{np.mean(tr_ac):.4f}", f"{np.mean(va_ac):.4f}",
                 f"{np.mean(accs):.4f}",  f"{np.mean(f1s):.4f}",
                 f"{np.mean(precs):.4f}", f"{np.mean(recs):.4f}",
                 f"{np.mean(gaps):+.4f}", ""])
    rows.append(["STD",
                 f"{np.std(tr_ac):.4f}",  f"{np.std(va_ac):.4f}",
                 f"{np.std(accs):.4f}",   f"{np.std(f1s):.4f}",
                 f"{np.std(precs):.4f}",  f"{np.std(recs):.4f}",
                 f"{np.std(gaps):.4f}", ""])

    print(tabulate(rows, headers=hdrs, tablefmt="double_grid"))

    mean_acc = float(np.mean(accs))
    mean_gap = float(np.mean(gaps))
    print(f"\n  Mean Test Accuracy : {mean_acc:.4f}  " +
          ("[TARGET MET >=90%]" if mean_acc >= 0.90
           else f"[NOT MET — gap {0.90-mean_acc:.4f}]"))
    print(f"  Mean Overfit Gap   : {mean_gap:+.4f}  " +
          ("[LOW]" if abs(mean_gap) < 0.05 else "[HIGH — overfitting]"))
    print("=" * W)

    return {
        "mean_accuracy":  mean_acc,
        "mean_f1":        float(np.mean(f1s)),
        "mean_precision": float(np.mean(precs)),
        "mean_recall":    float(np.mean(recs)),
        "mean_ovf_gap":   mean_gap,
    }


def save_results_csv(fold_results, output_dir=OUTPUT_DIR):
    df_r  = pd.DataFrame(fold_results)
    num_c = [c for c in df_r.columns if c != "Fold"]
    summ  = {"Fold": "MEAN+/-STD"}
    for c in num_c:
        summ[c] = f"{df_r[c].mean():.4f}+/-{df_r[c].std():.4f}"
    df_out = pd.concat([df_r, pd.DataFrame([summ])], ignore_index=True)
    path   = f"{output_dir}/siglip2_fakeddit_results.csv"
    df_out.to_csv(path, index=False)
    print(f"Results saved: {path}")
    try:
        display(df_out)
    except Exception:
        print(df_out.to_string(index=False))
    return path


def plot_visualizations(all_history, all_cms, output_dir=OUTPUT_DIR):
    # Training curves
    fig, axes = plt.subplots(2, NUM_FOLDS, figsize=(5*NUM_FOLDS, 8))
    C = ["#1976D2", "#E64A19"]
    for i, h in enumerate(all_history):
        ep = range(1, len(h["train_loss"]) + 1)
        ax = axes[0, i]
        ax.plot(ep, h["train_loss"], color=C[0], lw=2, label="Train")
        ax.plot(ep, h["val_loss"],   color=C[1], lw=2, ls="--", label="Val")
        ax.set_title(f"Fold {i+1} Loss", fontweight="bold")
        ax.legend(); ax.grid(alpha=0.3)
        ax = axes[1, i]
        ax.plot(ep, h["train_acc"], color=C[0], lw=2, label="Train")
        ax.plot(ep, h["val_acc"],   color=C[1], lw=2, ls="--", label="Val")
        ax.axhline(0.90, color="green", ls=":", lw=1.5, label="90% target")
        ax.set_title(f"Fold {i+1} Acc", fontweight="bold")
        ax.set_ylim(0.4, 1.0); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle("SigLIP-2 Fakeddit 2-Way — Training Curves",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    p1 = f"{output_dir}/training_curves.png"
    plt.savefig(p1, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved: {p1}")

    # Confusion matrices
    fig, axes = plt.subplots(1, NUM_FOLDS, figsize=(5*NUM_FOLDS, 5))
    for i, cm in enumerate(all_cms):
        pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        lbl = np.array([[f"{v}\n({p:.1f}%)" for v, p in zip(rv, rp)]
                        for rv, rp in zip(cm, pct)])
        sns.heatmap(cm, annot=lbl, fmt="", cmap="Blues", ax=axes[i],
                    xticklabels=["Real","Fake"], yticklabels=["Real","Fake"])
        acc_f = np.diag(cm).sum() / cm.sum()
        axes[i].set_title(f"Fold {i+1} (acc={acc_f:.3f})", fontweight="bold")
        axes[i].set_xlabel("Pred"); axes[i].set_ylabel("True")
    plt.suptitle("Confusion Matrices — 2-Way (Real vs Fake)",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    p2 = f"{output_dir}/confusion_matrices.png"
    plt.savefig(p2, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved: {p2}")

    # Overfitting gap bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    gaps   = [r["Overfit_Gap"] for r in FOLD_RESULTS]
    accs   = [r["Accuracy"]    for r in FOLD_RESULTS]
    colors = ["#43A047" if abs(g) < 0.05 else "#E53935" for g in gaps]
    bars   = ax.bar([f"F{r['Fold']}" for r in FOLD_RESULTS], gaps,
                    color=colors, edgecolor="black", alpha=0.85)
    ax.axhline( 0.05, color="orange", ls="--", lw=1.5, label="5% threshold")
    ax.axhline(-0.05, color="orange", ls="--", lw=1.5)
    ax.axhline( 0,    color="black",  lw=0.8)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{acc:.3f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_xlabel("Fold"); ax.set_ylabel("Overfitting Gap (Train - Test Acc)")
    ax.set_title("Overfitting Gap per Fold  (green=low, red=high)", fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    p3 = f"{output_dir}/overfitting_gap.png"
    plt.savefig(p3, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Saved: {p3}")


print("Results helpers ready  (FIX: table was truncated in original)")


Results helpers ready  (FIX: table was truncated in original)


In [10]:
# ── Cell 9: Load Fakeddit Data ────────────────────────────────────────────────
import time

print("Loading Fakeddit dataset (2-way, no metadata)...\n")
df_main = load_fakeddit(NUM_SAMPLES)
print(f"\nDataset shape: {df_main.shape}")
print(f"Label distribution:\n{df_main['label'].value_counts().to_string()}")
df_main.head(3)


Loading Fakeddit dataset (2-way, no metadata)...

Scanning for Fakeddit dataset...

Mounted datasets:
  /kaggle/input/datasets

Found (scan): /kaggle/input/datasets/vanshikavmittal/fakeddit-dataset/multimodal_only_samples/multimodal_test_public.tsv
  Columns: ['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score']...
  No image dir found — placeholder used for all samples

  Loaded 10,000 balanced samples  (Real=5000, Fake=5000)
  Images available: 0 / 10,000

Dataset shape: (10000, 4)
Label distribution:
label
0    5000
1    5000


,id,text,label,image_path
0,ccb7yxd,why cant we hold all these limes,0,None
1,bg64hn,goose chilling on a street light outside my of...,1,None
2,9b192a,missouri becomes first state to regulate use o...,1,None


In [ ]:
# ── Cell 10: Pre-cache Images & Run 5-Fold CV ─────────────────────────────────
# FIX: Images were opened from disk on every __getitem__ (blocking GPU).
#      Pre-cache all images into RAM once here.

print("Pre-caching images into RAM...")
t0 = time.time()
image_cache = build_image_cache(df_main)
print(f"Cache ready in {time.time()-t0:.1f}s  ({len(image_cache):,} images)\n")

print("Starting 5-fold cross-validation...\n")
fold_results, all_history, all_cms = run_kfold(df_main, image_cache)


Pre-caching images into RAM...
  Pre-caching 0 images into RAM...


0it [00:00, ?it/s]

  Cached 0 images in 0.0s  (0 failed)
Cache ready in 0.0s  (0 images)

Starting 5-fold cross-validation...

GPU at start: 15.5 / 15.6 GB free

Loading SigLIP-2 processor and backbone...


preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

  train:   0%|          | 0/1000 [00:00<?, ?it/s]

  E 4 [855s] | TrL=0.5466 TrA=0.8701 | VaL=0.4528 VaA=0.8400  <- best

  -- Epoch 5/12 --


  train:   0%|          | 0/1000 [00:00<?, ?it/s]

  E 5 [855s] | TrL=0.5179 TrA=0.9014 | VaL=0.4496 VaA=0.8500  <- best

  -- Epoch 6/12 --


  train:   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
# ── Cell 11: Display Results ──────────────────────────────────────────────────
summary = print_results_table(fold_results)
save_results_csv(fold_results)


In [ ]:
# ── Cell 12: Generate Visualizations ─────────────────────────────────────────
plot_visualizations(all_history, all_cms)


In [ ]:
# ── Cell 13: Final Summary ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  FINAL SUMMARY — SigLIP-2 | Fakeddit | 2-Way | No Metadata")
print("=" * 70)
print(f"  Model         : {SIGLIP_MODEL}")
print(f"  Samples       : {NUM_SAMPLES:,}  (balanced 50/50)")
print(f"  CV            : {NUM_FOLDS}-fold stratified")
print(f"  Split         : 80% train / 10% val / 10% test (per fold)")
print(f"  Eff. batch    : {BATCH_SIZE} x {ACCUM_STEPS} = {BATCH_SIZE*ACCUM_STEPS}")
print(f"  IMG_SIZE      : {IMG_SIZE}px  (FIX: was 384 -> 3x faster)")
print(f"  Image cache   : RAM-resident  (FIX: was per-step disk IO)")
print(f"  Grad.ckpt     : OFF  (FIX: was ON -> +30% overhead)")
print(f"  SWA           : GPU-resident  (FIX: was CPU/GPU each epoch)")
print(f"  SWA update_bn : added  (FIX: was MISSING)")
print(f"  Results table : fixed  (FIX: was truncated mid-line)")
print()
print("  -- Performance --")
mean_acc = summary["mean_accuracy"]
mean_gap = summary["mean_ovf_gap"]
print(f"  Mean Test Accuracy  : {mean_acc:.4f}")
print(f"  Mean Test F1 Macro  : {summary['mean_f1']:.4f}")
print(f"  Mean Test Precision : {summary['mean_precision']:.4f}")
print(f"  Mean Test Recall    : {summary['mean_recall']:.4f}")
print(f"  Mean Overfit Gap    : {mean_gap:+.4f}")
print()
acc_ok = mean_acc >= 0.90
gap_ok = abs(mean_gap) < 0.05
acc_msg = "MET" if acc_ok else "NOT MET  gap=" + str(round(0.90 - mean_acc, 4))
gap_msg = "MET" if gap_ok else "HIGH — overfitting"
print(f"  Target >=90% Acc    : [{acc_msg}]")
print(f"  Overfit Gap <5%     : [{gap_msg}]")
print()
print(f"  Files saved to {OUTPUT_DIR}/")
print(f"    siglip2_fakeddit_results.csv")
print(f"    training_curves.png")
print(f"    confusion_matrices.png")
print(f"    overfitting_gap.png")
print("=" * 70)
